# Code

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import math

# ============================================================
# CONFIG
# ============================================================
MODEL_PATH = 'face_landmarker.task'

# Índices de landmarks (modelo de 478 pontos)
RIGHT_IRIS = [469, 470, 471, 472]
RIGHT_IRIS_CENTER = 468
LEFT_IRIS = [474, 475, 476, 477]
LEFT_IRIS_CENTER = 473

# Pontos usados no cálculo de EAR (Eye Aspect Ratio) - detecção de piscada
# ordem: [canto externo, topo1, topo2, canto interno, base2, base1]
RIGHT_EYE_EAR = [33, 160, 158, 133, 153, 144]
LEFT_EYE_EAR = [362, 385, 387, 263, 373, 380]

# Cantos do olho usados para normalizar a posição da íris (gaze relativo)
RIGHT_EYE_CORNERS = (33, 133)   # (externo, interno)
LEFT_EYE_CORNERS = (362, 263)

EAR_BLINK_THRESHOLD = 0.21   # abaixo disso = olho fechado (ajuste testando)
EAR_CONSEC_FRAMES = 2        # nº de frames seguidos pra confirmar piscada
SMOOTHING_ALPHA = 0.4        # suavização do ponteiro (menor = mais suave)


# ============================================================
# FUNÇÕES AUXILIARES (branch: vision-pipeline / Pessoa A)
# ============================================================
def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])


def get_point(landmarks, idx, w, h):
    lm = landmarks[idx]
    return (lm.x * w, lm.y * h)


def calc_ear(landmarks, eye_indices, w, h):
    """Eye Aspect Ratio - quanto menor, mais fechado o olho está."""
    p = [get_point(landmarks, i, w, h) for i in eye_indices]
    vertical1 = euclidean(p[1], p[5])
    vertical2 = euclidean(p[2], p[4])
    horizontal = euclidean(p[0], p[3])
    if horizontal == 0:
        return 0.0
    return (vertical1 + vertical2) / (2.0 * horizontal)


def iris_center(landmarks, iris_indices, center_idx, w, h):
    """Média dos pontos do anel + centro, reduz ruído comparado a um ponto só."""
    pts = [get_point(landmarks, i, w, h) for i in iris_indices + [center_idx]]
    x = sum(p[0] for p in pts) / len(pts)
    y = sum(p[1] for p in pts) / len(pts)
    return (x, y)


def gaze_ratio(iris_pos, eye_corners, landmarks, w, h):
    """
    Posição da íris relativa aos cantos do olho (0 = canto externo, 1 = canto interno).
    Isso reduz a sensibilidade a pequenos movimentos de cabeça.
    """
    outer = get_point(landmarks, eye_corners[0], w, h)
    inner = get_point(landmarks, eye_corners[1], w, h)
    eye_width = euclidean(outer, inner)
    if eye_width == 0:
        return 0.5
    dist_from_outer = euclidean(iris_pos, outer)
    return dist_from_outer / eye_width  # ~0.0 a 1.0


class ExponentialSmoother:
    def __init__(self, alpha=SMOOTHING_ALPHA):
        self.alpha = alpha
        self.value = None

    def update(self, new_value):
        if self.value is None:
            self.value = new_value
        else:
            self.value = (
                self.alpha * new_value[0] + (1 - self.alpha) * self.value[0],
                self.alpha * new_value[1] + (1 - self.alpha) * self.value[1],
            )
        return self.value


# ============================================================
# SETUP
# ============================================================
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    print("Error: Could not open the camera.")
    exit()

left_iris_smoother = ExponentialSmoother()
right_iris_smoother = ExponentialSmoother()

# controle de piscada (branch: gesture-mapping / Pessoa B)
blink_counter = 0
blink_total = 0

print("Pressione ESC para sair.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    ts = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image, ts)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        # ---- ÍRIS (posição suavizada) ----
        raw_left = iris_center(landmarks, LEFT_IRIS, LEFT_IRIS_CENTER, w, h)
        raw_right = iris_center(landmarks, RIGHT_IRIS, RIGHT_IRIS_CENTER, w, h)
        left_pos = left_iris_smoother.update(raw_left)
        right_pos = right_iris_smoother.update(raw_right)

        cv2.circle(frame, (int(left_pos[0]), int(left_pos[1])), 3, (0, 255, 255), -1)
        cv2.circle(frame, (int(right_pos[0]), int(right_pos[1])), 3, (0, 255, 255), -1)

        # ---- GAZE RATIO (direção do olhar, 0=fora, 1=dentro) ----
        left_gaze = gaze_ratio(left_pos, LEFT_EYE_CORNERS, landmarks, w, h)
        right_gaze = gaze_ratio(right_pos, RIGHT_EYE_CORNERS, landmarks, w, h)
        avg_gaze = (left_gaze + right_gaze) / 2

        # ---- EAR / DETECÇÃO DE PISCADA ----
        left_ear = calc_ear(landmarks, LEFT_EYE_EAR, w, h)
        right_ear = calc_ear(landmarks, RIGHT_EYE_EAR, w, h)
        avg_ear = (left_ear + right_ear) / 2

        if avg_ear < EAR_BLINK_THRESHOLD:
            blink_counter += 1
        else:
            if blink_counter >= EAR_CONSEC_FRAMES:
                blink_total += 1
                # >>> AQUI é onde entraria o mapeamento gesto -> ação (Pessoa B)
                # ex.: pyautogui.click()
            blink_counter = 0

        # ---- Debug na tela ----
        cv2.putText(frame, f"EAR: {avg_ear:.2f}", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Gaze: {avg_gaze:.2f}", (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Piscadas: {blink_total}", (20, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow('Eye Tracking', frame)
    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()

Pressione ESC para sair.


KeyboardInterrupt: 